# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Justifier statistiquement l'usage de la **médiane (`PERCENTILE_APPROX`)** et de l'**écart interquartile (`IQR`)** face à l'asymétrie logistique.
2. Pratiquer les **fonctions de fenêtrage SQL** (`DENSE_RANK`, courbe cumulée de **Pareto**, `ROW_NUMBER`, `LAG`).
3. Produire les 3 tables décisionnelles de la couche **Gold** :
   - `/lakehouse/gold/kpi_categories`
   - `/lakehouse/gold/kpi_states`
   - `/lakehouse/gold/kpi_monthly`
4. Compacter la table Silver avec `OPTIMIZE`.

> **Auto-validation :** Lancez `task test:j3` pour vérifier vos tables Gold.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .getOrCreate()
)

spark.sql("""
    CREATE OR REPLACE TEMP VIEW v_delivered_orders AS
    SELECT *
    FROM delta.`/lakehouse/silver/orders`
    WHERE order_status = 'delivered'
      AND delivery_days IS NOT NULL
""")

## 1. Moyenne vs Médiane : Mesurer l'Asymétrie (`SKEWNESS`) des Délais

### Exercice 1.1 — Calculer les indicateurs de tendance centrale et de dispersion en Spark SQL
Écrivez une requête SQL sur `v_delivered_orders` retournant : `COUNT(*)`, `AVG(delivery_days)`, `STDDEV(delivery_days)`, `SKEWNESS(delivery_days)`, et les quantiles `P25`, `P50` (médiane), `P75`, `P95` via `PERCENTILE_APPROX(delivery_days, q)`.

In [ ]:
# TODO : Exécuter et afficher la requête SQL de diagnostic d'asymétrie sur `delivery_days`
df_skew = None
assert df_skew is not None, "TODO : Écrivez la requête SQL dans `df_skew`."
df_skew.show(truncate=False)

## 2. Table Gold #1 : Performance & Loi de Pareto par Catégorie (`/lakehouse/gold/kpi_categories`)

### Exercice 2.1 — Construire `/lakehouse/gold/kpi_categories`
Écrivez une requête Spark SQL (avec CTE `WITH cat_agg AS (...)`) sur `v_delivered_orders` (en filtrant les catégories ayant au moins `20` commandes via `HAVING COUNT(*) >= 20`) produisant **exactement** les colonnes suivantes (attendues par le Dashboard Streamlit) :
- `category`
- `total_orders` (`COUNT(*)`)
- `total_revenue` (`ROUND(SUM(total_amount), 2)`)
- `mean_basket`, `median_basket`
- `mean_delivery_days`, `median_delivery_days`
- `iqr_delivery_days` (`P75 - P25` sur `delivery_days`)
- `late_rate_pct` (`ROUND(100.0 * AVG(is_late), 2)`)
- `avg_review_score`
- `revenue_rank` (`DENSE_RANK() OVER (ORDER BY total_revenue DESC)`)
- `cumulative_revenue_pct` : part cumulée en % du chiffre d'affaires (`SUM(total_revenue) OVER (ORDER BY total_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(total_revenue) OVER ()`)

Sauvegardez le résultat avec `.coalesce(1)` (pour éviter le *Small Files Problem* sur HDFS !) au format **Delta** dans **`/lakehouse/gold/kpi_categories`**.

In [ ]:
# TODO : Écrire la requête SQL `df_gold_categories` et sauvegarder dans `/lakehouse/gold/kpi_categories`
df_gold_categories = None
assert df_gold_categories is not None, "TODO : Construisez `df_gold_categories`."

...
df_gold_categories.show(10, truncate=False)

## 3. Table Gold #2 : Disparités Géographiques par État (`/lakehouse/gold/kpi_states`)

### Exercice 3.1 — Construire `/lakehouse/gold/kpi_states` avec `ROW_NUMBER() OVER (PARTITION BY ...)`
Pour chaque `customer_state` non nul, calculez :
- `total_orders`, `total_revenue`, `freight_share_pct` (part moyenne des frais de port dans `total_amount` en %), `median_delivery_days`, `p90_delivery_days`, `late_rate_pct`, `avg_review_score`
- `top_category` : la catégorie n°1 en volume de commandes dans cet État, identifiée via `ROW_NUMBER() OVER (PARTITION BY customer_state ORDER BY COUNT(*) DESC) = 1`.

Sauvegardez avec `.coalesce(1)` au format **Delta** dans **`/lakehouse/gold/kpi_states`**.

In [ ]:
# TODO : Construire `df_gold_states` et sauvegarder dans `/lakehouse/gold/kpi_states`
df_gold_states = None
assert df_gold_states is not None, "TODO : Construisez `df_gold_states`."

...
df_gold_states.show(10, truncate=False)

## 4. Table Gold #3 : Série Mensuelle & Croissance MoM (`/lakehouse/gold/kpi_monthly`)

### Exercice 4.1 — Calculer la croissance mensuelle avec `LAG()` et compacter Silver avec `OPTIMIZE`
1. Agrégez `v_delivered_orders` par `order_month` (`HAVING COUNT(*) >= 50`) avec les colonnes : `order_month`, `total_orders`, `late_orders` (`SUM(is_late)`), `late_rate_pct`, `total_revenue`, `median_delivery_days`, `avg_review_score`.
2. Ajoutez `revenue_mom_growth_pct` en comparant `total_revenue` au mois précédent via `LAG(total_revenue, 1) OVER (ORDER BY order_month)`.
3. Sauvegardez avec `.coalesce(1)` dans **`/lakehouse/gold/kpi_monthly`**.
4. Exécutez enfin `OPTIMIZE delta.'/lakehouse/silver/orders'` pour compacter les petits fichiers Parquet issus du Jour 2.

In [ ]:
# TODO : Construire `df_gold_monthly`, sauvegarder dans `/lakehouse/gold/kpi_monthly`, puis lancer OPTIMIZE
df_gold_monthly = None
assert df_gold_monthly is not None, "TODO : Construisez `df_gold_monthly`."

...